# TODO

- [ ] Fetch ConEdison ArcGis Substation Level System Data
  - [EV Feeders](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-EV-Feeders)
  - [Segmented Hosting Capacity](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Segmented-Hosting-Capacity)
- [ ] Fetch ConEdison ArcGis Network data
  - [Networks](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Networks)
  - [Network MS Plates](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Network-MS-Plates)
  - [Network Structures](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Network-Structures)
  - [Network Storage HC](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Network-Storage-HC)
  - [Network Supplied Area](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Network-Supplied-Areas)
  - [Non-Wires Solutions Networks](../docs/Con%20Edison%20Hosting%20Capacity%20Web%20Application/rest-api.md#ConEd-Non-Wires-Solutions-Networks)


In [2]:
from datetime import date
from pathlib import Path
from tqdm.auto import tqdm
import geopandas as gpd
import pandas as pd
import json
import requests

In [ ]:
ROOT = Path("..").resolve()
BASE_URL = "https://services.arcgis.com/ciPnsNFi1JLWVjva/ArcGIS/rest/services/IEDR_CECONY/FeatureServer"
DATA_DIR = ROOT / "data/raw"
LAYER_NAMES = {
    0: "conedison_charge_capacity_3ph_feeders.geojson",
    1: "conedison_discharge_capacity_3ph_feeders.geojson",
    2: "conedison_no_charging_capacity_1ph_2ph_feeders.geojson",
    3: "conedison_transformers_460v_summer_capacity.geojson",
    4: "conedison_transformers_208v_summer_capacity.geojson",
    5: "conedison_transformers_460v_winter_capacity.geojson",
    6: "conedison_transformers_208v_winter_capacity.geojson",
    7: "conedison_summer_load_capacity_3ph_feeders.geojson",
    8: "conedison_winter_load_capacity_3ph_feeders.geojson",
    9: "conedison_no_load_capacity_1ph_2ph_feeders.geojson",
    10: "conedison_hosting_capacity_3ph_segments.geojson",
    11: "conedison_no_hosting_capacity_1ph_2ph_segments.geojson",
    12: "conedison_substation_level_system_data.geojson",
    13: "conedison_feeders_33kv.geojson",
    14: "conedison_network_structures.geojson",
    15: "conedison_network_primary_ess_hc.geojson",
    16: "conedison_network_ms_plates.geojson",
    17: "conedison_network_supplied_area.geojson",
    18: "conedison_network_boundary.geojson",
}

def fetch_feature_count(layer_id: int) -> int:
    response = requests.get(
        f"{BASE_URL}/{layer_id}/query",
        params={
            "where": "1=1",
            "returnCountOnly": "true",
            "f": "json",
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["count"]

def fetch_layer_features(
    layer_id: int,
    feature_count: int | None = None,
    batch_size: int = 1000,
    **query_params,
) -> tuple[gpd.GeoDataFrame, dict]:
    if feature_count is None:
        feature_count = fetch_feature_count(layer_id)

    batch_size = min(batch_size, 1000)
    source_url = f"{BASE_URL}/{layer_id}/query"
    source_params = []
    features = []
    
    base_params = {
        "where": "1=1",
        "outFields": "*",
        "orderByFields": "OBJECTID ASC",
        "returnGeometry": "true",
        "f": "geojson",
        **query_params,
    }

    with tqdm(
        total=feature_count,
        desc=f"Fetching layer {layer_id}",
        unit="feature",
    ) as progress:
        for offset in range(0, feature_count, batch_size):
            params = {
                **base_params,
                # Keep pagination controlled by this function.
                "resultOffset": offset,
                "resultRecordCount": batch_size,
            }
            source_params.append(params)
            response = requests.get(
                source_url,
                params=params,
                timeout=120,
            )
            response.raise_for_status()

            payload = response.json()

            if "error" in payload:
                raise RuntimeError(payload["error"])

            batch_features = payload.get("features", [])
            features.extend(batch_features)
            progress.update(len(batch_features))
            
    if features:
        gdf = gpd.GeoDataFrame.from_features(
            features,
            crs="EPSG:4326",
        )
    else:
        gdf = gpd.GeoDataFrame(
            {"geometry": []},
            geometry="geometry",
            crs="EPSG:4326",
        )        
            
    metadata = {
        "fetched_date": date.today().isoformat(),
        "source_url": source_url,
        "source_params": source_params,
        "row_count": len(gdf),
    }

    return gdf, metadata
    
def get_cache_paths(layer_id: int) -> tuple[Path, Path]:
    layer_name = LAYER_NAMES.get(layer_id)
    data_path = DATA_DIR / layer_name
    metadata_path = DATA_DIR / f"{layer_name}.meta.json"
    return data_path, metadata_path

def get_layer(
    layer_id: int,
    **query_params,
) -> gpd.GeoDataFrame:
    data_path, metadata_path = get_cache_paths(layer_id)

    if data_path.exists():
        return gpd.read_file(data_path)

    gdf, metadata = fetch_layer_features(layer_id, **query_params)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(data_path, driver="GeoJSON")

    metadata_path.write_text(json.dumps(metadata, indent=2))

    return gdf

charge_capacity_3ph_feeders_gdf = get_layer(0)
discharge_capacity_3ph_feeders_gdf = get_layer(1)
no_charging_capacity_1ph_2ph_feeders_gdf = get_layer(2)
transformers_460v_summer_capacity_gdf = get_layer(3)
transformers_208v_summer_capacity_gdf = get_layer(4)
transformers_460v_winter_capacity_gdf = get_layer(5)
transformers_208v_winter_capacity_gdf = get_layer(6)
summer_load_capacity_3ph_feeders_gdf = get_layer(7)
winter_load_capacity_3ph_feeders_gdf = get_layer(8)
no_load_capacity_1ph_2ph_feeders_gdf = get_layer(9)
hosting_capacity_3ph_segments_gdf = get_layer(10)
no_hosting_capacity_1ph_2ph_segments_gdf = get_layer(11)
substation_level_system_data_gdf = get_layer(12)
feeders_33kv_df = get_layer(13)
network_structures_gdf = get_layer(14, orderByFields="OBJECTID_1 ASC")
network_primary_ess_hc_gdf = get_layer(15, orderByFields="OBJECTID_1 ASC") # TODO: try retrieve from CECONY_Network_Storage_Prod for more features (5175)
network_ms_plates_gdf = get_layer(16)
network_supplied_area_gdf = get_layer(17)
network_boundary_gdf = get_layer(18)

In [22]:
charge_capacity_3ph_feeders_gdf["FEEDER_ID"]

0          ORIENT_17W41
1           OSSING_6W64
2                   9U3
3           MTKISC_7W40
4           MACQUE_9W69
              ...      
35878       MARYKL_7W47
35879              95U2
35880              62U2
35881    W_LACONIA_5X60
35882              91U3
Name: FEEDER_ID, Length: 35883, dtype: str

In [20]:
def normalize_feeder_id(values: pd.Series) -> pd.Series:
    return values.str.replace(r"^.*_", "", regex=True).str.zfill(5)

normalized_feeder_ids = (
    hosting_capacity_3ph_segments_gdf["FEEDER_ID"]
    .where(
        ~hosting_capacity_3ph_segments_gdf["FEEDER_ID"].str.contains("_"),
        normalize_feeder_id,
    )
)

pimba = network_primary_ess_hc_gdf[
    network_primary_ess_hc_gdf["Feeder1"].isin(normalized_feeder_ids)
    | network_primary_ess_hc_gdf["Feeder2"].isin(normalized_feeder_ids)
    | network_primary_ess_hc_gdf["Feeder3"].isin(normalized_feeder_ids)
]

pimba

,OBJECTID_1,GEOGRAPHIC,MSPlate,NetworkName,NetworkCode,Feeders_in_Plate,Feeder1,Feeder2,Feeder3,PeakN_0_or_PeakN_1,Shape__Area,Shape__Length,geometry
2,7,Westchester,122Q,Buchanan [Peekskill],wpk,6,13W88,13W83,13W78,3.7,0.000011,0.013481,"POLYGON ((-73.93521 41.27827, -73.93515 41.281..."
3,8,Westchester,31BZ,Harrison [Rye],whr,5,17W54,17W41,17W32,8.4,0.000011,0.013450,"POLYGON ((-73.74479 40.97681, -73.74472 40.980..."
5,12,Westchester,54CB,White Plains,wwp,2,08W79,08W76,0,0.2,0.000011,0.013458,"POLYGON ((-73.73632 41.05237, -73.73626 41.055..."
6,13,Westchester,122R,Buchanan [Peekskill],wpk,5,13W88,13W83,13W78,3.7,0.000011,0.013481,"POLYGON ((-73.93175 41.27824, -73.93169 41.281..."
8,20,Westchester,77BY,Pleasantville,wpl,1,14W03,0,0,0.0,0.000011,0.013466,"POLYGON ((-73.74506 41.12813, -73.74499 41.131..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2394,5218,Westchester,68BB,Harrison [Rye],whr,1,17W52,0,0,0.0,0.000011,0.013463,"POLYGON ((-73.81795 41.09938, -73.81789 41.102..."
2395,5220,Westchester,58BK,Grasslands,wgr,2,19W08,19W03,0,4.1,0.000011,0.013459,"POLYGON ((-73.79109 41.06617, -73.79102 41.069..."
2396,5221,Westchester,68AY,Grasslands,wgr,2,19W28,19W14,0,8.3,0.000011,0.013463,"POLYGON ((-73.82828 41.09949, -73.82822 41.102..."
2398,5224,Westchester,54CL,White Plains,wwp,2,08W79,08W95,0,0.6,0.000011,0.013458,"POLYGON ((-73.70537 41.05199, -73.7053 41.0552..."


In [23]:
pimba.explore()

In [21]:
network_primary_ess_hc_gdf

,OBJECTID_1,GEOGRAPHIC,MSPlate,NetworkName,NetworkCode,Feeders_in_Plate,Feeder1,Feeder2,Feeder3,PeakN_0_or_PeakN_1,Shape__Area,Shape__Length,geometry
0,2,Bronx,25J,Fordham,xfo,3,03X60,03X63,03X76,0.9,0.000010,0.012985,"POLYGON ((-73.89812 40.85401, -73.89826 40.856..."
1,4,Westchester,33AV,Granite hill [Yonkers],wyo,1,15W12,0,0,0.0,0.000011,0.013451,"POLYGON ((-73.84085 40.9845, -73.84079 40.9877..."
2,7,Westchester,122Q,Buchanan [Peekskill],wpk,6,13W88,13W83,13W78,3.7,0.000011,0.013481,"POLYGON ((-73.93521 41.27827, -73.93515 41.281..."
3,8,Westchester,31BZ,Harrison [Rye],whr,5,17W54,17W41,17W32,8.4,0.000011,0.013450,"POLYGON ((-73.74479 40.97681, -73.74472 40.980..."
4,9,Bronx,28H,Fordham,xfo,3,03X70,03X61,03X71,0.8,0.000010,0.012986,"POLYGON ((-73.90684 40.86072, -73.90698 40.863..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2395,5220,Westchester,58BK,Grasslands,wgr,2,19W08,19W03,0,4.1,0.000011,0.013459,"POLYGON ((-73.79109 41.06617, -73.79102 41.069..."
2396,5221,Westchester,68AY,Grasslands,wgr,2,19W28,19W14,0,8.3,0.000011,0.013463,"POLYGON ((-73.82828 41.09949, -73.82822 41.102..."
2397,5222,Westchester,48CP,White Plains,wwp,1,08W95,0,0,0.0,0.000011,0.013456,"POLYGON ((-73.69548 41.03213, -73.69541 41.035..."
2398,5224,Westchester,54CL,White Plains,wwp,2,08W79,08W95,0,0.6,0.000011,0.013458,"POLYGON ((-73.70537 41.05199, -73.7053 41.0552..."
